# FactorFolio Step 8: Risk Score Calibration

**AC420 project, Jasper Lin.** Build Guide Step 8 asks for a calibration check: score SPY alone, an all-utilities portfolio and a three-tech-stock portfolio, and make sure they come out in a sensible order. This notebook runs the app's own code (`core/`) on live Yahoo Finance data in Colab, so the scores match what the app shows.

It also checks the Step 8 "done when" rule for the suggestions engine: for a portfolio outside its comfort band, the top suggestion (added at 10%) should move the score into the band or toward it.

**To run:** Runtime, then Run all (about 2 minutes). No class files are needed. Written with AI assistance (Claude); see my AI usage log.

In [ ]:
!pip install -q --upgrade yfinance
!git clone -q https://github.com/jasperl12/AC420-project-FactorFolio-portfolio-risk-app.git factorfolio
import sys
sys.path.append('factorfolio')

import pandas as pd
from core import data as fdata, models, scoring, suggestions, projections

### Score the test portfolios
Each portfolio gets the homepage's risk score: beta, volatility and diversification ranked against the S&P 100 reference stocks, plus Altman Z health (utilities, real estate and financial firms are left out of Altman Z after calibration).

In [ ]:
test_portfolios = {
    'SPY alone': {'SPY': 100},
    'All utilities (DUK, SO, NEE)': {'DUK': 1, 'SO': 1, 'NEE': 1},
    'Three tech (AAPL, MSFT, NVDA)': {'AAPL': 1, 'MSFT': 1, 'NVDA': 1},
    'App sample portfolio': fdata.DEFAULT_PORTFOLIO,
    'KO + JNJ': {'KO': 1, 'JNJ': 1},
    'NVDA alone': {'NVDA': 1},
}

def score_portfolio(weights):
    ds = fdata.build_dataset(list(weights), weights=weights)
    if list(weights) == ['SPY']:
        res = models.analyze_stock(ds['window'], 'SPY')
        health = scoring.spy_health_points()
        return res, scoring.risk_score(res, health_points=health), None, None
    result = models.analyze_portfolio(ds, weights)
    held = list(result['weights'].index)
    prices = {t: ds['prices'][t].dropna().iloc[-1] for t in held}
    z_table = models.altman_table(held, result['weights'].to_dict(), prices)
    health = models.health_points(z_table)['points']
    return result['portfolio'], scoring.risk_score(result['portfolio'], health_points=health), result, z_table

rows = []
for name, weights in test_portfolios.items():
    res, scored, _, _ = score_portfolio(weights)
    parts = {c['name']: round(c['score']) for c in scored['components']}
    rows.append({'portfolio': name, 'score': round(scored['score'], 1), 'band': scored['band'],
                 'beta': round(res['beta'], 2), 'monthly vol': round(res['volatility'], 3),
                 '1 - R2': round(res['firm_specific'], 2), **parts})
calibration = pd.DataFrame(rows).set_index('portfolio')
calibration

In [ ]:
s = calibration['score']
print('SPY < three tech:', s['SPY alone'] < s['Three tech (AAPL, MSFT, NVDA)'])
print('Utilities < three tech:', s['All utilities (DUK, SO, NEE)'] < s['Three tech (AAPL, MSFT, NVDA)'])
print('KO + JNJ < sample < NVDA alone:', s['KO + JNJ'] < s['App sample portfolio'] < s['NVDA alone'])
assert s['Three tech (AAPL, MSFT, NVDA)'] > max(s['SPY alone'], s['All utilities (DUK, SO, NEE)'])
print('The three test portfolios score in a sensible order.')

### Suggestions check
For each portfolio and comfort level that is outside its band, show the top suggestion and the score after adding it at 10%.

In [ ]:
rf = projections.risk_free_rate()['rf']
checks = [('NVDA alone', 'Moderate'), ('Three tech (AAPL, MSFT, NVDA)', 'Conservative'), ('App sample portfolio', 'Moderate')]
for name, comfort in checks:
    res, scored, result, z_table = score_portfolio(test_portfolios[name])
    distance = suggestions.band_distance(scored['score'], comfort)
    if distance == 0:
        print(f'{name} ({comfort}): already in the band at {scored["score"]:.1f}')
        continue
    table = suggestions.test_scores(result['returns'], z_table, comfort, set(z_table['sector'].dropna()), rf)
    top = suggestions.suggest(table, scored['score'], comfort)
    if top.empty:
        print(f'{name} ({comfort}): no stock moves the score toward the band')
        continue
    best = top.iloc[0]
    where = 'into the band' if best['distance'] == 0 else 'toward the band'
    print(f"{name} ({comfort}, band {scoring.COMFORT_RANGES[comfort]}): {scored['score']:.1f} -> "
          f"{best['new_score']:.1f} by adding {top.index[0]} ({best['sector']}), {where}")
    assert best['distance'] < distance
print('Every top suggestion moves its portfolio into or toward the band.')